All the modules needed for RAG pipeline

In [ ]:
import os

#Modules for reading the files and ingesting
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Modules required for creating the vector database
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

#Files required for running the chatbot
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_groq import ChatGroq

In [ ]:
# Make your own groq key and store it in "key.txt" file
with open("key.txt", "r") as file:
    groq_key = file.read().strip()

os.environ["GROQ_API_KEY"] = groq_key

print("Groq API key loaded successfully.")

In [ ]:
#Codes for Ingesting the pdf
def ingest_documents(file_path):
    try:
        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            extracted = page.extract_text()

            if extracted:
                text += extracted + "\n"

        #print(text)

        # Split into chunks
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

        chunks = splitter.split_text(text)

        docs = [
            Document(
                page_content=chunk,
                metadata={"source": file_path}
            )
            for chunk in chunks
        ]

        return docs

    except Exception as e:
        print(f"Error while reading PDF: {e}")
        return []

#Code for building the vector db
def build_vector_db(docs, persist_directory="./chroma_db"):

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_db = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_directory
    )

    vector_db.persist()

    print(f"Vector database created and saved to {persist_directory}")

    return vector_db

In [ ]:
if __name__ == "__main__":
    folder_path = "PDFs"
    all_docs = []

    if os.path.exists(folder_path):
        # Iterate through every file in the folder
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            
            # Skip directories, only process files
            if os.path.isfile(file_path):
                print(f"Processing: {filename}...")
                docs = ingest_documents(file_path)
                if docs:
                    all_docs.extend(docs) # Add chunks to the master list
    else:
        print(f"Error: The folder '{folder_path}' was not found.")

    # Build the database only once with all collected documents
    if all_docs:
        print(f"Total chunks created: {len(all_docs)}")
        db = build_vector_db(all_docs)
    else:
        print("No documents were processed.")

Processing: module1-intro.pdf...
Processing: module10_ising.pdf...
Processing: module2-thermodynamics.pdf...
Processing: module3-ideal_gas_1.pdf...
Processing: module4-ideal_gas_II.pdf...
Processing: module5_canonical_ensemble.pdf...
Processing: module6_grand_canonical.pdf...
Processing: module7_quantum_statistics.pdf...
Processing: module8_fermigas.pdf...
Processing: module9_bosegas.pdf...
Total chunks created: 169


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6576.15it/s]


Vector database created and saved to ./chroma_db


C:\Users\alfie\AppData\Local\Temp\ipykernel_22168\823839577.py:52: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [ ]:
#Create the Groq Chatbot
# Recommended models: 'llama-3.3-70b-versatile' or 'mixtral-8x7b-32768'
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0
)

#Set up Memory and QA Chain(Stores Previous responses for better usability)
memory = ConversationBufferMemory(
    memory_key="chat_history", 
    return_messages=True
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(),
    memory=memory
)

while True:
    query = input("You: ")
    if query.lower() in ["exit", "quit"]: break
    
    result = qa_chain.invoke({"question": query})
    print(f"Groq AI: {result['answer']}")
    print("------------------")

Groq AI: The microcanonical ensemble is a statistical ensemble in physics that describes a system with a fixed energy, volume, and number of particles. It is also known as the "NVE ensemble", where:

* N is the number of particles
* V is the volume
* E is the energy

In a microcanonical ensemble, the system is isolated from its surroundings, meaning that it does not exchange energy or particles with the environment. The total energy of the system is fixed and is denoted by E.

The microcanonical ensemble is characterized by the following properties:

1. **Fixed energy**: The total energy of the system is fixed and is denoted by E.
2. **Fixed volume**: The volume of the system is fixed and is denoted by V.
3. **Fixed number of particles**: The number of particles in the system is fixed and is denoted by N.
4. **Isolation**: The system is isolated from its surroundings, meaning that it does not exchange energy or particles with the environment.

The microcanonical ensemble is used to des

KeyboardInterrupt: 